# Session 1: Single Spatial View

# What is **`vitessce`** ?

## Motivation

So far in the tutorial we have provided a general overview of Vitessce and introduced its key features:

- _Modular grid of views_
- _Supported data formats_
- _Coordinated interactivity_

Although Vitessce is very flexible, the internal JSON-based configuration is not ergonomic to write by hand and deploying a Vitessce-based visualization requires serving data files via a web server. Together, these challenges serve as a barrier to entry for some, and we were motivated to design a simplified API for computational biologists to visualize their own datasets with Vitessce. 

Enter **`vitessce`**...

## Overview

**`vitessce`** is a **Python package** designed to create interactive visualizations of single-cell and spatial omics data. Its main features include:

- Authoring Vitessce visualizations by configuring the data, views, and view linkages.

- Displaying Vitessce visualizations directly in computational notebooks (Jupyter, JupyterLab, Google Colab, Marimo)

- Transparently hosting local data files for visualizations (hiding web server complexities)


## How it works

The **`vitessce`** Python library exposes a simple API that maps directly to the internal Vitessce JSON configuration format. Users write Python programs with **`vitessce`** which ultimately:

- Emit JSON (the Vitessce visualization configuration)
- Render the visualization within computational notebooks as a widget 

# Getting started

The remainder of this notebook will focus on introducing the Vitessce Python package and its API.

Start by importing the `VitessceConfig` class from `vitessce`.

In [ ]:
!pip install "vitessce[all]==3.9.2" "scanpy" "easy-vitessce==0.0.12" "spatialdata==0.7.3" "spatialdata-plot==0.4.0"

In [ ]:
from vitessce import VitessceConfig

### 1. Introduction to Vitessce
Vitessce is an open-source framework for interactive visualization of spatial single-cell data. It uses a **VitessceConfig** class to define which data to load and how to display it in different 'views' (like a heatmap or a scatterplot).

### 2. Defining a configuration
We start by initializing a `VitessceConfig` object.
Then we call `.add_dataset()` and `.add_file()` (chained together) to define a new dataset and add a file to that dataset (that we just defined).

In [ ]:
from vitessce import VitessceConfig, ViewType as vt, DataType as dt, FileType as ft

# 1. Initialize the config with a name
config = VitessceConfig(schema_version="1.0.18", name="Single Spatial View Tutorial")

# 2. Add a dataset
# We'll use a public example dataset URL for this tutorial.
# Note that we use this `.add_file` method to specify a URL to a remotely-hosted data file.
dataset = config.add_dataset(name="Kuppe et al 2022").add_file(
    url="https://vitessce-data.storage.googleapis.com/0.0.33/main/kuppe-2022/kuppe_2022_nature.visium.ome.zarr",
    file_type="image.ome-zarr"
)

# 3. Add a Spatial View to the configuration
spatial_view = config.add_view("spatialBeta", dataset=dataset)

# 4. Arrange the layout (single view filling the whole widget)
config.layout(spatial_view)

print("Configuration created successfully!")

### 3. Rendering the Widget
Finally, we call `.widget()` to render the interactive dashboard directly in Colab.

In [ ]:
my_widget = config.widget()
my_widget

Notes

*   We used `.add_file` to specify a URL to a file that was already remotely hosted on the internet publicly. How might we specify a dataset that is on our local machine or on a connected compute cluster? ([docs](https://python-docs.vitessce.io/data_options.html))
*   Where can I find a list of the valid `file_type` values? ([docs](https://vitessce.io/docs/data-types-file-types/))

*   Where can I find a list of the valid `view_type` values? ([docs](https://vitessce.io/docs/components/))

* If you modify the configuration code, be sure to run BOTH the `config = VitessceConfig` and the `config.widget()` notebook cells (and any additional cells in between). If you are not sure why, please let us know and we can explain.



#### Looking under the hood: the initial JSON configuration

In [ ]:
# We can view the internal JSON representation of the configuration with the `.to_dict` method.
# (The `base_url` parameter allows for specifying a URL prefix, which is only relevant when using local data objects that do not yet have remote URLs)
config.to_dict(base_url="")

#### Looking under the hood: the current JSON configuration

In [ ]:
my_widget._config

In [ ]:
# Try it: Run the cell, then zoom in or out of the image in the widget above. Then re-run this code cell.
my_widget._config["coordinationSpace"]["spatialZoom"]["A"]

## Specifying initial visualization properties

By default, Vitessce fits the data (for spatial/imaging and scatterplot views) in the viewport on first load.
You can override such initial settings by specifying **initial coordination values**.

The concept of "coordination" comes from Vitessce's support for "coordinated multiple views", which means that visualization properties can be linked across views. The support for coordinated properties __also serves as the mechanism to specify initial visualization properties.__

Therefore, despite the current notebook focusing on the single-view case, we briefly introduce the concept of coordination as a prerequisite for specifying initial settings (or overriding their defaults).

Almost all of the state (visualization properties, interaction settings, etc.) in Vitessce are represented as variables in the **"coordination space"** part of the Vitessce configuration. Each of these variables has a type ("coordination type"), a name ("coordination scope"), and a value ("coordination value"). Think of it like defining variables in a statically-typed programming language:

```c
int x = 99;         // C variables
spatialZoom y = 63; // Vitessce coordination values (In reality, these are specified via a JSON representation)
```

Examples of coordination types used by the spatial view are:

| Coordination Type | Description | Example Value |
|---|---|---|
| `spatialZoom` | Zoom level (Higher = more zoomed-in; Log scale) | `2` |
| `spatialTargetX` | X-coordinate of the viewport center | `500.0` |
| `spatialTargetY` | Y-coordinate of the viewport center | `500.0` |
| `featureSelection` | List of selected genes or other features | `["EPCAM"]` |

Use `config.link_views_by_dict()` to set initial values for one or more coordination types across a list of views:

```python
config.link_views_by_dict([my_view],
    {
        "spatialZoom": 2,
        "spatialTargetX": 500.0,
        "spatialTargetY": 500.0
    }
)
```

In [ ]:
from vitessce import VitessceConfig

config2 = VitessceConfig(schema_version="1.0.18", name="Initial Properties Example")

dataset2 = config2.add_dataset(name="Kuppe et al 2022").add_file(
    url="https://vitessce-data.storage.googleapis.com/0.0.33/main/kuppe-2022/kuppe_2022_nature.visium.ome.zarr",
    file_type="image.ome-zarr"
)

spatial_view2 = config2.add_view("spatialBeta", dataset=dataset2)

# Set the initial zoom level and pan position
config2.link_views_by_dict([spatial_view2],
    {
        "spatialZoom": -2,
        "spatialTargetX": 300.0,
        "spatialTargetY": 300.0,
    }
)

config2.layout(spatial_view2)
config2.widget()

### Exercise 1

👉 **Modify the `config2` cell above** to:

- **Change** the `spatialZoom` value to `1` (zoomed in)
- **Change** `spatialTargetX` and `spatialTargetY` to pan to the center of the tissue
- **Disable** the R, G, B channel labels (hint: see `spatialChannelLabelsVisible`)

After making your changes, re-run the cell to see the updated widget.

Question: What does the "Click to recenter" button do now?

## Using local data with SpatialData

### What is SpatialData?

[SpatialData](https://spatialdata.scverse.org/) is a Python framework for storing and manipulating spatial omics data. A `SpatialData` object bundles together:

- **Images**: tissue photographs or microscopy images (stored as OME-Zarr)
- **Labels**: segmentation masks identifying individual cells (bitmask images)
- **Shapes**: geometric observations such as Visium spots (circles) or cell outlines (polygons)
- **Points**: individual molecule coordinates, e.g. from Xenium/MERFISH experiments
- **Tables**: AnnData objects linking observations (spots, cells) to gene expression data

Vitessce can read SpatialData objects via the `SpatialDataWrapper` class.

### Download the example Visium dataset

The cell below downloads a small Visium breast-cancer dataset as a `.zarr` zip archive and unzips it locally. The download only runs once — subsequent runs detect the existing directory and skip the download.

In [ ]:
import os
from os.path import join, isfile, isdir
from urllib.request import urlretrieve
import zipfile

data_dir = "data"
zip_filepath = join(data_dir, "visium.spatialdata.zarr.zip")
sdata_filepath = join(data_dir, "visium.spatialdata.zarr")

if not isdir(sdata_filepath):
    if not isfile(zip_filepath):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve(
            "https://data-2.vitessce.io/sdata-datasets/visium.spatialdata.zarr.zip",
            zip_filepath
        )
    with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
        zip_ref.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), sdata_filepath)

print("Dataset ready at:", sdata_filepath)

### Inspect the SpatialData object

Let's load the dataset and look at what it contains.

In [ ]:
from spatialdata import read_zarr

sdata = read_zarr(sdata_filepath)
sdata

SpatialData objects may not contain all 5 types of spatial elements. For example, this object only contains Images, Shapes, and Tables.

Question: how many coordinate systems does it contain?

Question: How do these elements relate to the Zarr folder contents?

In [ ]:
# os.listdir(join(sdata_filepath, "images"))

In [ ]:
# os.listdir(join(sdata_filepath, "shapes"))

In [ ]:
# os.listdir(join(sdata_filepath, "tables"))

In [ ]:
# os.listdir(join(sdata_filepath, "tables", "table"))

### Configure Vitessce for image + spots

The `SpatialDataWrapper` maps paths within the SpatialData Zarr store to Vitessce data types. We pass:

- `image_path`: the relative path to a particular image within the Zarr store
- `obs_spots_path`: the relative path to the shapes element that corresponds to the Visium spots (as circles)
- `obs_feature_matrix_path`: the relative path to the expression matrix array within the Zarr store (will always be within a Table element)
- `coordinate_system`: the coordinate system to use for alignment of the spatial layers

In [ ]:
from vitessce import (
    VitessceConfig,
    ViewType as vt,
    SpatialDataWrapper,
)

vc = VitessceConfig(schema_version="1.0.18", name="Visium Spatial Data")

wrapper = SpatialDataWrapper(
    sdata_store=sdata_filepath,
    image_path="images/ST8059050_hires_image",
    obs_spots_path="shapes/ST8059050",
    table_path="tables/table",
    obs_feature_matrix_path="tables/table/X",
    coordinate_system="ST8059050",
    coordination_values={
        "obsType": "spot" # Since this is a Visium dataset, we specify an alternative "obsType" (the default is "cell")
    },
)
dataset = vc.add_dataset(name="Visium Breast Cancer").add_object(wrapper)

# Add a spatial view, a layer controller, and a gene list
spatial = vc.add_view("spatialBeta", dataset=dataset)

# Link all views to share the same obsType coordination value
vc.link_views_by_dict([spatial], { "obsType": "spot" })

vc.layout(spatial)
vc.widget()

#### Exercise

- Try updating the SpatialDataWrapper parameters to view the other Visium sample in the dataset.

## Multiplexed imaging

So far we have used an RGB (e.g., histology) image. Vitessce also supports visualization of **multiplexed (multi-channel) images** in which each channel represents a different protein marker or staining. The individual channels can be thought of as greyscale images, but we use **pseudocoloring** (mapping each channel to a different color) to visualize multiple channels at a time.

When Vitessce detects that an image is a multiplex (non-RGB) image, it will assign a set of default colors to each channel. 
Vitessce will also initialize each channel's intensity window based on the distribution of intensity values (i.e., using the interquartile range).


Because Vitessce can display multiple images at a time (i.e., multiple image _layers_), the visualization properties for imaging data are represented using a hierarchical data structure.

The `link_views_by_dict()` method with the `CoordinationLevel` (CL) function allows you to specify these types of hierarchical properties in the initial configuration.

The example below uses a [CyCIF](https://www.cycif.org/) multiplexed immunofluorescence dataset (outputs of the [MCMICRO](https://mcmicro.org/) pipeline). We configure the initial channel colors.

Similar to `VitessceConfig.link_views`, the first parameter of `VitessceConfig.link_views_by_dict` is an array of views for which you want to specify the coordination values. The second parameter of `link_views_by_dict` is a Python dictionary mapping coordination types to their initial values.

In [ ]:
mcmicro_zip = join(data_dir, "mcmicro_io.spatialdata.zarr.zip")
mcmicro_path = join(data_dir, "mcmicro_io.spatialdata.zarr")

if not isdir(mcmicro_path):
    if not isfile(mcmicro_zip):
        os.makedirs(data_dir, exist_ok=True)
        urlretrieve(
            "https://data-2.vitessce.io/sdata-datasets/mcmicro_io.spatialdata.zarr.zip",
            mcmicro_zip
        )
    with zipfile.ZipFile(mcmicro_zip, "r") as zip_ref:
        zip_ref.extractall(data_dir)
        os.rename(join(data_dir, "data.zarr"), mcmicro_path)

sdata_mc = read_zarr(mcmicro_path)
sdata_mc

In [ ]:
from vitessce import (
    VitessceConfig,
    ViewType as vt,
    CoordinationLevel as CL,
    SpatialDataWrapper,
    get_initial_coordination_scope_prefix,
)

vc_mc = VitessceConfig(schema_version="1.0.18", name="Multiplexed Image (CyCIF)")

mc_wrapper = SpatialDataWrapper(
    sdata_store=mcmicro_path,
    image_path="images/exemplar-001_image",
    coordinate_system="global",
    coordination_values={
        "obsType": "cell"
    },
)
dataset_mc = vc_mc.add_dataset(name="MCMicro CyCIF").add_object(mc_wrapper)

spatial_mc = vc_mc.add_view("spatialBeta", dataset=dataset_mc)

# Set initial channel colors and visibility
vc_mc.link_views_by_dict(
    [spatial_mc],
    {
        "imageLayer": CL([{
            "photometricInterpretation": "BlackIsZero",
            "imageChannel": CL([
                {"spatialTargetC": 0, "spatialChannelColor": [255, 0, 0],   "spatialChannelOpacity": 1.0},
                {"spatialTargetC": 1, "spatialChannelColor": [0, 255, 0],   "spatialChannelOpacity": 1.0},
                {"spatialTargetC": 2, "spatialChannelColor": [0, 0, 255],   "spatialChannelOpacity": 1.0},
            ]),
        }]),
    },
    scope_prefix=get_initial_coordination_scope_prefix("A", "image"),
)

vc_mc.layout(spatial_mc)
vc_mc.widget()

#### Exercise

Now, try modifying the above configuration to specify better channel intensity window ranges. (Hint: try `spatialChannelWindow`)

## More spatial layers

We often want to render multiple data layers rendered simultaneously: the tissue image, cell segmentation outlines, spot circles, and molecule point clouds. Vitessce supports all of these in the same spatial view via the `layerControllerBeta`.

The table below shows the `SpatialDataWrapper` parameter for each layer type:

| Layer Type | What it shows | `SpatialDataWrapper` param |
|---|---|---|
| Image | Tissue photograph or multiplex image | `image_path` |
| Spots | Visium capture spots (circles) | `obs_spots_path` |
| Segmentation polygons | Cell outlines (GeoDataFrame shapes) | `obs_shapes_path` |
| Segmentation bitmask | Integer label image where each value = one cell | `obs_segmentations_path` |
| Molecule points | Sub-cellular transcript coordinates | `obs_points_path` |

The [SpatialData Blobs](https://spatialdata.scverse.org/en/stable/api/datasets.html#spatialdata.datasets.blobs) toy dataset bundles several of these together and is useful for learning. Let's build a multi-layer view with it.

In [ ]:
import spatialdata
import pandas as pd
from anndata import AnnData

# Create the blobs dataset in memory
sdata_blobs = spatialdata.datasets.blobs()
sdata_blobs

Create a `Table` with a `var` dataframe corresponding to the `genes` column of the Points table

In [ ]:
ddf = sdata_blobs.points['blobs_points']
ddf

In [ ]:
# The current `table` element contains a `var` dataframe corresponding
# to the 3 image channels, rather than the 2 gene IDs.
print(sdata_blobs.tables['table'].var.index.tolist())

In [ ]:
# We need to create another table which has a var.index column containing the gene IDs for the points.
unique_gene_ids = ddf["genes"].unique().compute().tolist()
points_var_df = pd.DataFrame(index=unique_gene_ids, data=[], columns=[])
points_table = AnnData(var=points_var_df, obs=None, X=None)
sdata_blobs.tables['table_points'] = points_table

In [ ]:
unique_gene_ids

In [ ]:
sdata_blobs.write('data/blobs.sdata.zarr', overwrite=True)

In [ ]:
sdata_blobs

Note the addition of the `table_points` element in the updated SpatialData object.

In [ ]:
from vitessce import SpatialDataWrapper, CoordinationLevel as CL, get_initial_coordination_scope_prefix

vc_blobs = VitessceConfig(schema_version="1.0.18", name="Blobs: Image + Labels + Points")

# Wrapper 1: image + segmentation bitmask (labels)
blobs_wrapper = SpatialDataWrapper(
    sdata_store="data/blobs.sdata.zarr",
    # The following paths are relative to the root of the SpatialData zarr store on-disk.
    image_path="images/blobs_multiscale_image",
    obs_segmentations_path="labels/blobs_labels",
    obs_embedding_paths=["tables/table/obsm/X_umap"],
    obs_feature_matrix_path="tables/table/X",
    coordinate_system="global",
    coordination_values={
        "obsType": "blob",
        "featureType": "channel",
        "fileUid": "my_unique_id"
    }
)
# Wrapper 2: molecule points
points_wrapper = SpatialDataWrapper(
    sdata_store="data/blobs.sdata.zarr",
    # The following paths are relative to the root of the SpatialData zarr store on-disk.
    obs_points_path="points/blobs_points",
    obs_feature_matrix_path="tables/table_points/X", # TODO
    coordinate_system="global",
    coordination_values={
        "obsType": "point",
        "featureType": "gene",
        "fileUid": "other_unique_id"
    }
)
dataset_blobs = vc_blobs.add_dataset(name="Blobs").add_object(blobs_wrapper).add_object(points_wrapper)

spatial_blobs = vc_blobs.add_view("spatialBeta", dataset=dataset_blobs)

vc_blobs.link_views_by_dict([spatial_blobs], {
    'imageLayer': CL([{
        "fileUid": "my_unique_id",
        'photometricInterpretation': 'BlackIsZero',
        'spatialLayerOpacity': 0.9,
        'imageChannel': CL([
            {
                "spatialTargetC": 0,
                "spatialChannelColor": [255, 0, 0],
                "spatialChannelOpacity": 1.0,
                "spatialChannelWindow": [0.0, 1.0],
            },
            {
                "spatialTargetC": 1,
                "spatialChannelColor": [0, 255, 0],
                "spatialChannelOpacity": 1.0,
                "spatialChannelWindow": [0.0, 1.0],
            },
            {
                "spatialTargetC": 2,
                "spatialChannelColor": [0, 0, 255],
                "spatialChannelOpacity": 1.0,
                "spatialChannelWindow": [0.0, 1.0],
            }
        ])
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "image"))

vc_blobs.link_views_by_dict([spatial_blobs], {
    'segmentationLayer': CL([{
        "fileUid": "my_unique_id",
        'segmentationChannel': CL([{
            'spatialChannelVisible': True,
            'obsType': 'blob',
            "featureType": "channel",
        }]),
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "obsSegmentations"))

vc_blobs.link_views_by_dict([spatial_blobs], {
    'pointLayer': CL([{
        "fileUid": "other_unique_id",
        "obsType": "point",
        "featureType": "gene",
        "obsHighlight": None,
        "obsColorEncoding": "randomByFeature",
        "spatialLayerOpacity": 1.0,
    }]),
}, scope_prefix=get_initial_coordination_scope_prefix("A", "obsPoints"))

vc_blobs.layout(spatial_blobs)
vc_blobs.widget()

## Simplifying things with SpatialData-Plot and EasyVitessce

The low-level `vitessce` API is powerful but verbose. [EasyVitessce](https://vitessce.github.io/easy_vitessce/) intercepts [SpatialData-Plot](https://spatialdata.scverse.org/projects/plot/en/latest/) plotting calls and automatically generates an interactive Vitessce widget instead of returning a static figure.

### How it works

1. Install and **`import easy_vitessce as ev`**. Running this import line itself __automatically__ enables the interactive output (with the option to disable later if needed)
2. Use the familiar `sdata.pl.render_*().pl.show()` chain. (If unfamiliar with SpatialData-Plot, see its [documentation](https://spatialdata.scverse.org/projects/plot/en/latest/))
3. The widget is live: pan, zoom, select cells, change channels

This means existing SpatialData-Plot workflows become interactive with just one extra import.

In [ ]:
import easy_vitessce as ev
import spatialdata_plot

# This line is required when the notebook kernel is running on a different machine (e.g., Google Colab, Docker container, or HPC cluster)
ev.config.set({ 'data.wrapper_param_suffix': '_store' })

# Re-use the Visium SpatialData object loaded earlier
(
    sdata
        .pl.render_images("ST8059050_hires_image")
        .pl.render_shapes("ST8059050", color="Fth1", table_name="table")
        .pl.show("ST8059050")
)

Compared to the `SpatialDataWrapper` example earlier, the EasyVitessce version is much more concise. The parameters map directly to SpatialData-Plot conventions:

- `.pl.render_images("element_name")`: add an image layer
- `.pl.render_shapes("element_name", color="GeneName", table_name="table")`: add a spot or segmentation layer colored by gene expression
- `.pl.show("coordinate_system")` — render and display the widget

If you are curious, you can check out the EasyVitessce [source code](https://github.com/vitessce/easy_vitessce/blob/main/src/easy_vitessce/spatialdata_plot.py) which internally translates the SpatialData-Plot syntax to the Vitessce configuration code we learned above.

### Static vs interactive

You can switch between static matplotlib output and the interactive Vitessce widget at any time:

In [ ]:
# Disable interactive plots: renders with static matplotlib
ev.disable_plots(["spatialdata-plot"])

(
    sdata
        .pl.render_images("ST8059050_hires_image")
        .pl.render_shapes("ST8059050", color="Fth1", table_name="table")
        .pl.show("ST8059050")
)

In [ ]:
# Re-enable interactive plots for the exercise below
ev.enable_plots(["spatialdata-plot"])

### Exercise 2

👉 **Modify the EasyVitessce cell above** to:

- **Change** the `color` parameter from `"Fth1"` to a different gene (Note: this is a mouse dataset)
- **Add** a segmentation label layer using `.pl.render_labels("element_name")` before `.pl.show()`


In [ ]:
sdata.tables["table"].var

In [ ]:
# Your answer here
(
    sdata
        .pl.render_images("ST8059050_hires_image")
        .pl.render_shapes("ST8059050", color="red", table_name="table")
        .pl.show("ST8059050")
)

Finally, check out the EasyVitessce [documentation](https://vitessce.github.io/easy_vitessce/easy_vitessce.html#easy_vitessce.spatialdata_plot.VitesscePlotAccessor.render_shapes) to learn more about the available functions and parameters.